## Regime Validation Against Market Structure (Regime vs Structure)

### Goal
Validate that HMM regimes **explain structural breaks** (not just visual segmentation) by linking each regime to:
- **Rolling correlation** (relationship strength / breakdown)
- **Clustering fragmentation / persistence** (network structure stability)
- **PC1 dominance / instability** (market tightness / common factor dominance)

This notebook follows the rubric exactly:

1) **Overlay regimes** on:
   - rolling correlations
   - clustering fragmentation / persistence
   - PC1 dominance / instability

2) **Quantitative validation** (by regime):
   - mean rolling correlation
   - cluster persistence (and/or fragmentation)
   - PC1 variance explained

3) **Hypothesis check** (explicit YES/NO):
   - Do relationships weaken in stress regimes?
   - Does PC1 dominate during stress?
   - Do clusters fragment during stress?

---

### Inputs (Dependencies)
(expected under `results/`):
- Rolling correlation artifacts:
  - `rolling_corr*.csv`, `rolling_correlation*.csv`, `pairwise_rolling_corr*.csv`
- Clustering artifacts:
  - `cluster_persistence*.csv`, `co_cluster*.csv`, `cluster_fragmentation*.csv`, `n_clusters*.csv`
- PCA artifact:
  - `pca*_pc1*_metrics.csv` (PC1 variance / dominance proxy)

- `regime_probabilities.csv` containing `regime_most_likely`

---

### Definitions (What each metric means)

**Rolling correlation (relationship strength)**
- We reduce pairwise rolling correlations into a **single daily scalar**:
  - `rolling_corr_mean` = average correlation across pairs that day
- Lower in stress can indicate **breakdown**; higher indicates **tightening**.
  - We will test direction explicitly.

**Cluster persistence / fragmentation**
- If we have a persistence series, we use:
  - `cluster_persistence` (higher = more stable structure)
- If we have a fragmentation series, we use:
  - `cluster_fragmentation` or `n_clusters` (higher = more fragmented)

**PC1 variance (dominance)**
- `pc1_variance` as proxy for market tightness / dominance of common factor.
  - Higher = more “one-factor” market.

---

### Hypotheses (Explicit Gates)

H1 — Relationships weaken in stress regimes  
We test if rolling correlation **decreases** in stress vs normal:
- YES if (stress mean < normal mean) and p < 0.05  
- NO otherwise (and we state the direction)

H2 — PC1 dominates during stress  
We test if PC1 variance **increases** in stress vs normal:
- YES if (stress mean > normal mean) and p < 0.05  
- NO otherwise

H3 — Clusters fragment during stress  
We test one of:
- Fragmentation metric increases in stress (YES if stress > normal and p < 0.05)
- OR persistence decreases in stress (YES if stress < normal and p < 0.05)

---


In [ ]:
import importlib
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path

current_dir = Path(os.getcwd())
project_root = current_dir.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.regimes.regime_validation as rv
importlib.reload(rv)

from src.regimes.regime_validation import (
    merge_regime_and_structure, 
    calculate_regime_stats, 
    test_hypothesis, 
    plot_regime_boxplots,
    load_optional_structure_series,
    hypothesis_yesno,
    pick_stress_and_normal_regimes
)

results_dir = project_root / "results"

# Load Regime Probabilities (contains 'regime_most_likely')
# Using index_col=0 to automatically set the Date index
df_regimes = pd.read_csv(
    results_dir / "regime_probabilities.csv", 
    index_col=0, 
    parse_dates=True
)

# Load PCA Metrics (contains PC1 Variance)
df_pca = pd.read_csv(
    results_dir / "pca_5Y_rolling_pc1_metrics.csv", 
    index_col=0, 
    parse_dates=True
)

# Rename the first column to 'pc1_variance' for consistency
pc1_col = df_pca.columns[0]
pc1_var = df_pca[[pc1_col]].rename(columns={pc1_col: 'pc1_variance'})

# --- 2. Data Alignment & Preprocessing ---
# Ensure both indices are datetime objects
df_regimes.index = pd.to_datetime(df_regimes.index)
pc1_var.index = pd.to_datetime(pc1_var.index)

# Remove timezone information to ensure compatibility during merge
# This prevents mismatches between timezone-aware and timezone-naive dates
df_regimes.index = df_regimes.index.tz_localize(None)
pc1_var.index = pc1_var.index.tz_localize(None)

# Sort indices to align temporal order
df_regimes = df_regimes.sort_index()
pc1_var = pc1_var.sort_index()

# --- 3. Merge Datasets ---
# Inner join on index to align regimes with structural metrics
df_master = merge_regime_and_structure(df_regimes[['regime_most_likely']], pc1_var)

In [ ]:
opt = load_optional_structure_series(results_dir)

print("Rolling corr status:", opt.get("_rolling_corr_status"))
print("Cluster status:", opt.get("_cluster_status"))
print("Rolling corr file:", opt.get("_rolling_corr_file"))
print("Cluster file:", opt.get("_cluster_file"))

added_cols = []

if opt.get("rolling_corr_mean") is not None:
    df_master = df_master.join(opt["rolling_corr_mean"].rename("rolling_corr_mean"), how="inner")
    added_cols.append("rolling_corr_mean")

if opt.get("cluster_persistence") is not None:
    df_master = df_master.join(opt["cluster_persistence"].rename("cluster_persistence"), how="inner")
    added_cols.append("cluster_persistence")

if opt.get("cluster_fragmentation") is not None:
    df_master = df_master.join(opt["cluster_fragmentation"].rename("cluster_fragmentation"), how="inner")
    added_cols.append("cluster_fragmentation")

print("Added optional structure columns:", added_cols)
print(df_master.head())

In [ ]:
print(f"Indices in Regimes: {len(df_regimes)}")
print(f"Indices in PCA: {len(pc1_var)}")
print(f"Overlapping Indices: {len(df_master)}")

if df_master.empty:
    print("Error: No overlapping dates found. Please verify the date ranges in the input files.")
    print(f"Regime Range: {df_regimes.index.min()} to {df_regimes.index.max()}")
    print(f"PCA Range: {pc1_var.index.min()} to {pc1_var.index.max()}")
else:
    print("Merge successful. Preview of master dataframe:")
    print(df_master.head())

In [ ]:
#  Statistical Validation of Regimes Against Structural Metrics

# 1. Metric Definition
# PC1 variance is our primary proxy for systemic risk (market tightness).
# Cluster stability is included conditionally if the data is available.
metrics = ["pc1_variance"]

if "rolling_corr_mean" in df_master.columns:
    metrics.append("rolling_corr_mean")

if "cluster_persistence" in df_master.columns:
    metrics.append("cluster_persistence")

if "cluster_fragmentation" in df_master.columns:
    metrics.append("cluster_fragmentation")

if "cluster_stability" in df_master.columns:
    metrics.append("cluster_stability")

# 2. Descriptive Statistics
# Group by regime to establish baseline behaviors (Mean/Std) for each state.
stats_summary = calculate_regime_stats(df_master, metrics)

print("--- Structural Statistics by Regime ---")
print(stats_summary)
stats_summary.to_csv(results_dir / "regime_structure_comparison.csv")

# 3. Regime Identification (Crisis vs. Normal)
# We programmatically identify regimes to ensure consistent labeling:
# - Crisis: Highest PC1 variance (assets move together, high correlation).
# - Normal: Lowest PC1 variance (assets move independently, high diversification).
crisis_regime = stats_summary['pc1_variance']['mean'].idxmax()
normal_regime = stats_summary['pc1_variance']['mean'].idxmin()

print(f"\nAuto-Detected Regimes:")
print(f" - CRISIS Regime: {crisis_regime} (High Systemic Risk)")
print(f" - NORMAL Regime: {normal_regime} (Low Systemic Risk)")

# 4. Hypothesis Testing
print("\n--- Hypothesis Testing Results ---")

# Hypothesis 1: Does Market Tightness (PC1) significantly increase in stress regimes?
# Expectation: Crisis Mean > Normal Mean with statistical significance.
h1 = test_hypothesis(df_master, 'pc1_variance', crisis_regime, normal_regime)

is_pc1_valid = h1.get('significant', False) and h1.get('diff', 0) > 0

print("1. Test: PC1 Dominance (Market Tightening) in Stress")
print(f"   Verdict: {'PASS' if is_pc1_valid else 'FAIL'}")

print(
    f"   (Stress Mean: {h1.get('stress_mean', np.nan):.4f}, "
    f"Normal Mean: {h1.get('normal_mean', np.nan):.4f}, "
    f"Diff: {h1.get('diff', np.nan):.6f}, "
    f"p={h1.get('p_value', np.nan):.4e})"
)

# Hypothesis 2: Does Cluster Structure significantly change during stress?
# We test for any structural break (fragmentation or consolidation) in asset clusters.
if 'cluster_stability' in df_master.columns:
    h2 = test_hypothesis(df_master, 'cluster_stability', crisis_regime, normal_regime)
    print(f"\n2. Test: Cluster Stability Change in Stress")
    print(f"   Verdict: {'PASS' if h2['significant'] else 'FAIL'}")
    print(f"   (Crisis: {h2['crisis_mean']:.3f} vs Normal: {h2['normal_mean']:.3f})")

# 5. Visualization
plot_regime_boxplots(df_master, metrics)

In [ ]:
# --- Explicit YES/NO answers to hypotheses ---

stress_regime, normal_regime = pick_stress_and_normal_regimes(stats_summary, pc1_metric="pc1_variance")
print(f"Stress Regime={stress_regime}, Normal Regime={normal_regime}")

rows = []

# Q1: Do relationships weaken in stress regimes?
# Expect rolling correlation to DROP in stress (stress < normal)
if "rolling_corr_mean" in df_master.columns:
    r = hypothesis_yesno(df_master, "rolling_corr_mean", stress_regime, normal_regime, direction="lt")
    rows.append({"question": "Do relationships weaken in stress regimes?", **r})
else:
    rows.append({"question": "Do relationships weaken in stress regimes?", "answer": "NOT AVAILABLE", "reason": "missing rolling_corr_mean"})

# Q2: Does PC1 dominate during stress?
# Expect PC1 variance to RISE in stress (stress > normal)
r = hypothesis_yesno(df_master, "pc1_variance", stress_regime, normal_regime, direction="gt")
rows.append({"question": "Does PC1 dominate during stress?", **r})

# Q3: Do clusters fragment during stress?
# If fragmentation exists -> stress higher
# If persistence exists -> stress lower (less persistent => more fragmentation)
if "cluster_fragmentation" in df_master.columns:
    r = hypothesis_yesno(df_master, "cluster_fragmentation", stress_regime, normal_regime, direction="gt")
    rows.append({"question": "Do clusters fragment during stress?", **r})
elif "cluster_persistence" in df_master.columns:
    r = hypothesis_yesno(df_master, "cluster_persistence", stress_regime, normal_regime, direction="lt")
    rows.append({"question": "Do clusters fragment during stress?", **r})
else:
    rows.append({"question": "Do clusters fragment during stress?", "answer": "NOT AVAILABLE", "reason": "missing cluster metric"})

df_answers = pd.DataFrame(rows)
print("\n--- Hypothesis Check (YES/NO) ---")
display(df_answers)

df_answers.to_csv(results_dir / "regime_hypothesis_answers.csv", index=False)
print("Saved: results/regime_hypothesis_answers.csv")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Enhanced Visualization: Overlay Regimes on a Single Timeline Plot
plt.figure(figsize=(15, 6))


plt.plot(df_master.index, df_master['pc1_variance'], 
         color='black', linestyle='--', linewidth=1, label='Market Tightness (PC1)', zorder=1)


regimes = sorted(df_master['regime_most_likely'].unique())
colors = sns.color_palette("bright", len(regimes))

for regime, color in zip(regimes, colors):
    regime_mask = df_master['regime_most_likely'] == regime
    
    
    plt.scatter(df_master.index[regime_mask], 
                df_master.loc[regime_mask, 'pc1_variance'], 
                s=15, color=color, label=f'Regime {regime}', zorder=2)


plt.title("Regime Classification Over Time (PC1 Variance)", fontsize=14)
plt.xlabel("Date")
plt.ylabel("PC1 Variance (Market Tightness)")
plt.legend(loc='upper left', frameon=True, fancybox=True, framealpha=0.9)
plt.grid(True, alpha=0.3)
plt.tight_layout()


plt.show()

In [ ]:
# Overlay regimes on rolling corr + clustering
import matplotlib.pyplot as plt
import seaborn as sns

regimes = sorted(df_master["regime_most_likely"].unique())
colors = sns.color_palette("bright", len(regimes))

# 1) Rolling correlation overlay
if "rolling_corr_mean" in df_master.columns:
    plt.figure(figsize=(15, 5))
    plt.plot(df_master.index, df_master["rolling_corr_mean"],
             color="black", linestyle="--", linewidth=1, label="rolling_corr_mean", zorder=1)

    for r, c in zip(regimes, colors):
        mask = df_master["regime_most_likely"] == r
        plt.scatter(df_master.index[mask], df_master.loc[mask, "rolling_corr_mean"],
                    s=12, color=c, label=f"Regime {r}", zorder=2)

    plt.title("Overlay: Rolling Correlation Mean by Regime", fontsize=14)
    plt.xlabel("Date")
    plt.ylabel("Mean Rolling Correlation")
    plt.grid(True, alpha=0.3)
    plt.legend(loc="upper left", frameon=True)
    plt.tight_layout()
    plt.show()

# 2) Clustering overlay (fragmentation OR persistence)
cluster_metric = None
cluster_title = None

if "cluster_fragmentation" in df_master.columns:
    cluster_metric = "cluster_fragmentation"
    cluster_title = "Overlay: Cluster Fragmentation by Regime (higher = more fragmented)"
elif "cluster_persistence" in df_master.columns:
    cluster_metric = "cluster_persistence"
    cluster_title = "Overlay: Cluster Persistence by Regime (lower = more fragmented)"

if cluster_metric is not None:
    plt.figure(figsize=(15, 5))
    plt.plot(df_master.index, df_master[cluster_metric],
             color="black", linestyle="--", linewidth=1, label=cluster_metric, zorder=1)

    for r, c in zip(regimes, colors):
        mask = df_master["regime_most_likely"] == r
        plt.scatter(df_master.index[mask], df_master.loc[mask, cluster_metric],
                    s=12, color=c, label=f"Regime {r}", zorder=2)

    plt.title(cluster_title, fontsize=14)
    plt.xlabel("Date")
    plt.ylabel(cluster_metric)
    plt.grid(True, alpha=0.3)
    plt.legend(loc="upper left", frameon=True)
    plt.tight_layout()
    plt.show()

### Calculate volatility of each regime to confirm the true "Stress/Shock" regime

In [ ]:

df_features = pd.read_csv(results_dir / "regime_features.csv", index_col=0, parse_dates=True)

df_vol_analysis = df_features.join(df_regimes[['regime_most_likely']], how='inner')

# Calculate "Realized Volatility" (Absolute Daily Move)
vol_stats = df_vol_analysis.groupby('regime_most_likely')['move_chg'].apply(
    lambda x: pd.Series({
        'avg_daily_move_bp': x.abs().mean(),  
        'volatility_of_vol': x.abs().std(),   
        'max_move_bp': x.abs().max(),         
        'count': len(x)
    })
).unstack()

print("Realized Volatility by Regime")
print(vol_stats)


# Statistical Regime Validation

Validate that HMM regimes correspond to distinct structural market behavior.

---

## Structural Statistics by Regime (PC1 Variance)

| Regime | Count | Mean PC1 | Std Dev |
|--------|-------|----------|---------|
| 0 | 1152 | 0.4185 | 0.0771 |
| 1 | 1215 | **0.4219 (Highest)** | 0.0741 |
| 2 | 1168 | **0.4151 (Lowest)** | 0.0774 |
| 3 | 319 | 0.4188 | 0.0747 |

---

## Auto-Detected Regimes

- **Stress Regime (Correlation Stress): Regime 1**
- **Normal Regime (Risk-On): Regime 2**

Stress regime identified as highest PC1 dominance.

---

## Hypothesis Testing Results

### H1 — Do relationships weaken in stress regimes?

**NOT AVAILABLE**

Rolling correlation series was not daily-indexed and could not be merged.
Pair-level rolling metrics require aggregation before regime comparison.

---

### H2 — Does PC1 dominate during stress?

**YES**

- Stress Mean = **42.19%**
- Normal Mean = **41.51%**
- Difference = +0.68%
- p-value = **0.0283**

There is a statistically significant increase in common factor dominance during stress regimes.

This confirms that stress regimes are characterized by stronger systemic co-movement.

---

### H3 — Do clusters fragment during stress?

**NOT AVAILABLE**

Cluster persistence file is pair-level and not daily-indexed.
A daily structural fragmentation metric is required for validation.

---

## Realized Volatility by Regime

| Regime | Avg Daily Move (bps) | Vol-of-Vol | Max Move (bps) | Count |
|--------|---------------------|------------|----------------|-------|
| 0 | 1.62 | 1.52 | 9.44 | 1456 |
| 1 | 1.48 | 1.42 | 8.54 | 1537 |
| 2 | 1.56 | 1.48 | 9.46 | 1490 |
| 3 | **6.19** | **5.01** | **28.23** | 364 |

---

## Interpretation

The 4-State HMM distinguishes between:

### Regime 1 — Correlation Stress
- Highest PC1 dominance
- Low realized volatility
- Represents tight, grinding markets

### Regime 3 — Volatility Shock
- Massive realized volatility spike (4x baseline)
- Not the highest PC1
- Represents shock events rather than systemic tightening

This separation between correlation stress and volatility shock is economically meaningful and valuable for Relative Value positioning.

---

## Conclusion

The regime model successfully captures:

- Structural tightening (PC1 dominance)
- Distinct volatility shock regimes

However, rolling correlation and clustering fragmentation validation require daily-aggregated structural metrics before hypothesis testing.